# Lab 6 - Interactive Visual Analytics with Folium

In [1]:
import folium, pandas as pd
from folium.plugins import MarkerCluster
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians
df = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
sites = df.groupby('Launch Site', as_index=False).first()[['Launch Site','Lat','Long']]
sites

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


### Mark all launch sites on the map

In [2]:
site_map = folium.Map(location=[29.559, -95.083], zoom_start=4)
for _, r in sites.iterrows():
    folium.Circle([r['Lat'], r['Long']], radius=1000, color='#d35400', fill=True).add_child(folium.Popup(r['Launch Site'])).add_to(site_map)
    label = '<div style="font-size:12px;color:#d35400;"><b>' + str(r['Launch Site']) + '</b></div>'
    folium.map.Marker([r['Lat'], r['Long']], icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0), html=label)).add_to(site_map)
site_map

### Mark the success/failed launches for each site

In [3]:
df['marker_color'] = df['class'].apply(lambda c: 'green' if c==1 else 'red')
cluster_map = folium.Map(location=[29.559, -95.083], zoom_start=4)
mc = MarkerCluster().add_to(cluster_map)
for _, r in df.iterrows():
    folium.Marker([r['Lat'], r['Long']], icon=folium.Icon(color='white', icon_color=r['marker_color'])).add_to(mc)
cluster_map

### Calculate the distance from a launch site to the coastline

In [4]:
def calc_dist(lat1, lon1, lat2, lon2):
    R = 6373.0
    dlon = radians(lon2)-radians(lon1); dlat = radians(lat2)-radians(lat1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    return R*2*atan2(sqrt(a), sqrt(1-a))

site_lat, site_lon = 28.563197, -80.576820
coast_lat, coast_lon = 28.56367, -80.57163
d = calc_dist(site_lat, site_lon, coast_lat, coast_lon)
print('distance to coastline (km):', round(d, 2))
prox = folium.Map(location=[site_lat, site_lon], zoom_start=14)
folium.Marker([site_lat, site_lon], popup='CCAFS SLC-40').add_to(prox)
lbl = '<div style="font-size:12px;color:#c0392b;"><b>' + str(round(d,2)) + ' KM</b></div>'
folium.Marker([coast_lat, coast_lon], icon=DivIcon(icon_size=(20,20), icon_anchor=(0,0), html=lbl)).add_to(prox)
folium.PolyLine([[site_lat, site_lon],[coast_lat, coast_lon]], weight=2).add_to(prox)
prox

distance to coastline (km): 0.51
